# **NLP**

1. Natural Language Processing (NLP) is a subfield of Artificial Intelligence focused on enabling machines to understand, interpret, generate, and interact using human language

2. Types:
- Stateless: Learns on random portions of text at each iteration, without any information on the rest of the text.
- Statefull: Preserves the hidden state between training iterations and continues reading where it left off, allowing it to learn longer patterns

# **Generating Shakespearean Text Using a Character RNN**

In [2]:
import tensorflow as tf

shakespeare_url = "https://homl.info/shakespeare"
filepath = tf.keras.utils.get_file("shakespeare.txt", shakespeare_url)
with open(filepath) as f:
    shakespeare_text = f.read()

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
print(shakespeare_text[:80])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.


In [4]:
# all distinct charecters
"".join(sorted(set(shakespeare_text.lower())))

"\n !$&',-.3:;?abcdefghijklmnopqrstuvwxyz"

In [5]:
text_vec_layer = tf.keras.layers.TextVectorization(split="character", standardize="lower")
text_vec_layer.adapt([shakespeare_text])
encoded = text_vec_layer([shakespeare_text])[0]

In [6]:
encoded -= 2  # drop tokens 0 (pad) and 1 (unknown), which we will not use
n_tokens = text_vec_layer.vocabulary_size() - 2  # number of distinct chars = 39
dataset_size = len(encoded)  # total number of chars = 1,115,394

In [7]:
n_tokens

39

In [8]:
dataset_size

1115394

In [9]:
def to_dataset(seq, len, shuffle=False, seed=None, batch_size=32):
    ds = tf.data.Dataset.from_tensor_slices(seq)
    ds = ds.window(len + 1, shift=1, drop_remainder=True)
    ds = ds.flat_map(lambda window_ds:window_ds.batch(len + 1))
    if shuffle:
        ds = ds.shuffle(100_000, seed=seed)
    ds = ds.batch(batch_size)
    return ds.map(lambda window: (window[:, :-1], window[:, 1:])).prefetch(1)

In [10]:
# extra code – a simple example using to_dataset()
# There's just one sample in this dataset: the input represents "to b" and the output represents "o be"
list(to_dataset(text_vec_layer(["To be"])[0], len=4))

[(<tf.Tensor: shape=(1, 4), dtype=int64, numpy=array([[ 4,  5,  2, 23]])>,
  <tf.Tensor: shape=(1, 4), dtype=int64, numpy=array([[ 5,  2, 23,  3]])>)]

In [11]:
length = 100
tf.random.set_seed(42)
train_set = to_dataset(encoded[:1_000_000], len=length, shuffle=True,
                       seed=42)
valid_set = to_dataset(encoded[1_000_000:1_060_000], len=length)
test_set = to_dataset(encoded[1_060_000:], len=length)

In [12]:
train_set

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, None), dtype=tf.int64, name=None), TensorSpec(shape=(None, None), dtype=tf.int64, name=None))>

## **Building and Training the Char-RNN Model**

In [14]:
tf.random.set_seed(42)

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=n_tokens, output_dim=16),
    tf.keras.layers.GRU(128, return_sequences=True),
    tf.keras.layers.Dense(n_tokens, activation="softmax")
])
model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam",
              metrics=["accuracy"])
model_ckpt = tf.keras.callbacks.ModelCheckpoint(
    "my_shakespeare_model.keras", monitor="val_accuracy", save_best_only=True)
history = model.fit(train_set, validation_data=valid_set, epochs=10,
                    callbacks=[model_ckpt])

Epoch 1/10
  31245/Unknown 416s 13ms/step - accuracy: 0.5446 - loss: 1.5078

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


31247/31247 ━━━━━━━━━━━━━━━━━━━━ 434s 13ms/step - accuracy: 0.5757 - loss: 1.3844 - val_accuracy: 0.5315 - val_loss: 1.6097
Epoch 2/10
31247/31247 ━━━━━━━━━━━━━━━━━━━━ 450s 13ms/step - accuracy: 0.5971 - loss: 1.2929 - val_accuracy: 0.5393 - val_loss: 1.5773
Epoch 3/10
31247/31247 ━━━━━━━━━━━━━━━━━━━━ 441s 13ms/step - accuracy: 0.6007 - loss: 1.2761 - val_accuracy: 0.5410 - val_loss: 1.5781
Epoch 4/10
31247/31247 ━━━━━━━━━━━━━━━━━━━━ 442s 13ms/step - accuracy: 0.6028 - loss: 1.2665 - val_accuracy: 0.5430 - val_loss: 1.5673
Epoch 5/10
31247/31247 ━━━━━━━━━━━━━━━━━━━━ 484s 15ms/step - accuracy: 0.6039 - loss: 1.2606 - val_accuracy: 0.5458 - val_loss: 1.5601
Epoch 6/10
31247/31247 ━━━━━━━━━━━━━━━━━━━━ 423s 13ms/step - accuracy: 0.6052 - loss: 1.2558 - val_accuracy: 0.5462 - val_loss: 1.5636
Epoch 7/10
31247/31247 ━━━━━━━━━━━━━━━━━━━━ 419s 13ms/step - accuracy: 0.6063 - loss: 1.2513 - val_accuracy: 0.5452 - val_loss: 1.5642
Epoch 8/10
31247/31247 ━━━━━━━━━━━━━━━━━━━━ 422s 13ms/step - accur

In [20]:
shakespeare_model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Lambda(lambda X: X - 2),  # no <PAD> or <UNK> tokens
    model
])

In [40]:
X_new = tf.constant(["how are yo"])
y_proba = shakespeare_model.predict(X_new, verbose=0)[0, -1]

y_pred = tf.argmax(y_proba).numpy()
predicted_char = text_vec_layer.get_vocabulary()[y_pred + 2]
predicted_char

np.str_('u')

## **Generating Fake Shakespearean Text**

In [41]:
log_probas = tf.math.log([[0.5, 0.4, 0.1]])  # probas = 50%, 40%, and 10%
tf.random.set_seed(42)
tf.random.categorical(log_probas, num_samples=8)  # draw 8 samples

<tf.Tensor: shape=(1, 8), dtype=int64, numpy=array([[0, 0, 1, 1, 1, 0, 0, 0]])>

In [48]:
def next_char(text, temperature=1):
    y_proba = shakespeare_model.predict([text], verbose=0)[0, -1:]
    rescaled_logits = tf.math.log(y_proba) / temperature
    char_id = tf.random.categorical(rescaled_logits, num_samples=1)[0, 0]
    return text_vec_layer.get_vocabulary()[char_id + 2]

In [64]:
def extend_text(text, n_chars=50, temperature=1):
    for _ in range(n_chars):
        text += next_char(text, temperature)
    return text.numpy()[0].decode("utf-8")

In [65]:
tf.random.set_seed(42)

In [68]:
text = tf.constant(["To be or not to be"])

print(extend_text(text, temperature=0.01))

To be or not to be a strangely.

provost:
i will not be a strangely 


In [66]:
print(extend_text(text, temperature=1))

To be or not to be the kingsh:
here changevost you. believe yourself


In [70]:
print(extend_text(text, temperature=100))

To be or not to beq'
n?
nms3enp,pmtde'
,o :?sdoqyqtov!l&b!je?$;hiqte
